# Notebook 09: Efficient Attention Mechanisms

## Why This Matters

The standard attention mechanism has $O(N^2)$ time and memory complexity in sequence length $N$. For $N=4096$, the attention matrix alone is $4096 \times 4096 \times 4 \text{ bytes} = 64 \text{ MB}$ per layer per batch. This is why GPT-3 was limited to 2048 tokens and why scaling to 100K+ token contexts requires architectural innovation. FlashAttention is the single most important inference optimization in the last five years — it's used in virtually every production LLM serving system. Multi-Query Attention and Grouped-Query Attention reduce KV cache memory by 8-32x, which is the binding constraint for long-context inference batch size. This notebook builds MQA and GQA from scratch and benchmarks memory vs sequence length.

## Background

Two distinct problems plague standard attention at scale:

1. **Memory wall:** The $N \times N$ attention matrix must be materialized in HBM (GPU memory), limiting batch size and sequence length.

2. **KV cache bottleneck:** Each attention head in each layer requires its own K and V matrices in the cache. For LLaMA-7B (32 layers, 32 heads), the KV cache for a 4K context at FP16 is ~2GB. This limits how many requests can be served concurrently.

**Three complementary solutions:**
- **FlashAttention:** Avoids materializing the full $N \times N$ matrix by tiling the computation across SRAM
- **Multi-Query Attention (MQA):** All query heads share a single K,V head → 32x KV cache reduction
- **Grouped-Query Attention (GQA):** Groups of query heads share K,V heads → tunable tradeoff between MQA (max efficiency) and MHA (max quality)

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
torch.manual_seed(42)
print("Ready.")

## 1. FlashAttention: IO-Aware Tiling (Narrative)

Standard attention has a hidden cost beyond FLOPs: **memory bandwidth**. The $N \times N$ attention matrix must be written to HBM (GPU high-bandwidth memory, ~1-2 TB/s) and read back for the softmax and value multiplication. For large $N$, this HBM traffic dominates total runtime.

**FlashAttention's core insight** (Dao et al. 2022): Never materialize the full $N \times N$ attention matrix. Instead, tile the computation across SRAM (fast on-chip memory, ~19 TB/s) by computing softmax and weighted value sums in blocks.

**The algorithm (mental model):**
1. Divide Q, K, V into blocks that fit in SRAM (typically 64-128 rows)
2. For each block of Q, iterate over all blocks of K, V:
   - Compute partial scores $QK^T / \sqrt{d_k}$ (fits in SRAM)
   - Maintain running softmax numerics (log-sum-exp trick for numerical stability)
   - Accumulate weighted value sum
3. Only write the final output to HBM — the intermediate attention matrix never touches HBM

**Result:** 
- Same numerical output as standard attention (up to floating point precision)
- Memory complexity: $O(N)$ not $O(N^2)$ — the attention matrix never exists in full
- 2-4x faster wall-clock time for long sequences due to reduced HBM traffic
- Enables sequence lengths of 4K-128K that were previously impossible

**FlashAttention 2 & 3** (2023-2024): Further optimizations for A100/H100: better parallelization across attention heads, reduced non-matmul FLOPs, and support for variable-length sequences. FlashAttention-3 on H100 achieves ~75% of theoretical FLOP utilization.

## 2. Multi-Query Attention (MQA)

In standard MHA, each of the $H$ heads has its own $W^K$ and $W^V$ projection. In **Multi-Query Attention** (Shazeer 2019):

- $H$ query heads: each head still has its own $W^Q$ projection
- **1 key head** and **1 value head**: all query heads share a single $W^K$ and $W^V$

$$Q_h = x W^Q_h \quad \text{(h = 1...H separate projections)}$$
$$K = x W^K \quad V = x W^V \quad \text{(single shared projection)}$$

**KV cache impact:** MQA reduces KV cache size by factor $H$ (number of heads). For LLaMA-7B with 32 heads, this is a 32× reduction — from 2GB to 62.5MB per sequence!

**Quality tradeoff:** MQA reduces model capacity slightly. For some tasks (especially retrieval-heavy), quality degrades noticeably. The "sweet spot" is GQA (below).

**Used in:** PaLM, Falcon, early versions of code models. Gradually superseded by GQA.

In [ ]:
class MultiQueryAttention(nn.Module):
    """
    Multi-Query Attention: H query heads, but only 1 K and 1 V head.
    All query heads share the same K, V projections.
    """
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # H separate query projections
        self.W_q = nn.Linear(d_model, d_model, bias=False)   # (d_model → n_heads * d_k)
        # Single shared K and V projections
        self.W_k = nn.Linear(d_model, self.d_k, bias=False)  # (d_model → d_k) — just 1 head
        self.W_v = nn.Linear(d_model, self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, _ = x.shape

        # Q: (B, n_heads, T, d_k)
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        # K, V: (B, 1, T, d_k) — single head, broadcast across all query heads
        K = self.W_k(x).unsqueeze(1)  # (B, 1, T, d_k)
        V = self.W_v(x).unsqueeze(1)

        # Attention: Q (B, h, T, d_k) × K (B, 1, T, d_k) → broadcast works
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)  # (B, h, T, T)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        # (B, h, T, T) × (B, 1, T, d_k) → (B, h, T, d_k)
        out = torch.matmul(self.dropout(attn), V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)

# Compare parameter counts
d_model, n_heads = 512, 8
d_k = d_model // n_heads

mha_kv_params = 2 * d_model * d_model  # K and V projections in MHA
mqa_kv_params = 2 * d_model * d_k      # K and V in MQA (1 head each)
print(f"MHA K+V params: {mha_kv_params:,}")
print(f"MQA K+V params: {mqa_kv_params:,}")
print(f"Reduction: {mha_kv_params/mqa_kv_params:.0f}x fewer K,V params")

# KV cache size comparison
T, n_layers, dtype_bytes = 4096, 32, 2  # 4K context, 32 layers, FP16
mha_kv_cache = 2 * n_layers * n_heads * T * d_k * dtype_bytes / 1e9
mqa_kv_cache = 2 * n_layers * 1 * T * d_k * dtype_bytes / 1e9
print(f"\nKV cache for T={T}, L={n_layers}, {n_heads} heads, FP16:")
print(f"  MHA: {mha_kv_cache:.3f} GB")
print(f"  MQA: {mqa_kv_cache:.3f} GB ({mha_kv_cache/mqa_kv_cache:.0f}x smaller)")

## 3. Grouped-Query Attention (GQA)

GQA (Ainslie et al. 2023) is a generalization that interpolates between MHA and MQA:
- Divide the $H$ query heads into $G$ groups
- Each group shares one K, V head
- $G = H$: standard MHA; $G = 1$: MQA

$$Q_h, \quad K_{\lfloor h \cdot G/H \rfloor}, \quad V_{\lfloor h \cdot G/H \rfloor}$$

**Why GQA is the current standard:**
- **LLaMA-2:** Uses GQA with $G = 8$ heads for 70B model (4 K,V heads vs 64 query heads = 8× KV cache reduction)
- **Mistral-7B:** 8 query heads per KV head (GQA-8)
- **LLaMA-3:** All sizes use GQA

GQA was found to recover most of MHA quality while providing most of MQA's efficiency gains. The optimal $G$ depends on the model size: larger models need more K,V heads to maintain quality.

In [ ]:
class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention: n_kv_heads key/value heads shared among n_heads query heads.
    n_kv_heads=n_heads: standard MHA
    n_kv_heads=1:       MQA
    n_kv_heads=G:       GQA with G groups
    """
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads  # How many Q heads share each KV head
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, n_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        B, T, _ = x.shape

        # Q: (B, n_heads, T, d_k)
        Q = self.W_q(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        # K, V: (B, n_kv_heads, T, d_k)
        K = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)

        # Repeat K, V for each query group: (B, n_heads, T, d_k)
        K = K.repeat_interleave(self.n_rep, dim=1)
        V = V.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(self.dropout(attn), V)
        out = out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        return self.W_o(out)

# Test all three variants
d_model, n_heads = 512, 8
B, T = 4, 64

mha = GroupedQueryAttention(d_model, n_heads, n_kv_heads=8)    # Standard MHA
gqa = GroupedQueryAttention(d_model, n_heads, n_kv_heads=2)    # GQA: 4 Q heads per KV
mqa = GroupedQueryAttention(d_model, n_heads, n_kv_heads=1)    # MQA

x = torch.randn(B, T, d_model)
print("Output shapes:")
for name, attn in [("MHA (8 KV)", mha), ("GQA (2 KV)", gqa), ("MQA (1 KV)", mqa)]:
    out = attn(x)
    kv_params = sum(p.numel() for n, p in attn.named_parameters() if '_k' in n or '_v' in n)
    print(f"  {name}: output={out.shape}, KV params={kv_params:,}")

## 4. Benchmarking Memory vs Sequence Length

In [ ]:
def estimate_attention_memory(seq_len, n_heads, d_k, n_kv_heads, dtype_bytes=2):
    """Estimate peak attention memory in MB."""
    # Attention score matrix: (n_heads, T, T)
    attn_matrix_mb = n_heads * seq_len * seq_len * dtype_bytes / 1e6
    # KV cache: (n_kv_heads, T, d_k) * 2 (K and V)
    kv_cache_mb = 2 * n_kv_heads * seq_len * d_k * dtype_bytes / 1e6
    return attn_matrix_mb, kv_cache_mb

seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]
n_heads = 32
d_k = 128  # LLaMA-7B style

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for variant, n_kv, color, ls in [
    ("MHA (32 KV)", 32, 'blue', '-'),
    ("GQA-8 (4 KV)", 4, 'green', '--'),
    ("GQA-32 (1 KV / MQA)", 1, 'red', ':'),
]:
    attn_mems = []
    kv_mems = []
    for T in seq_lengths:
        attn_mb, kv_mb = estimate_attention_memory(T, n_heads, d_k, n_kv)
        attn_mems.append(attn_mb)
        kv_mems.append(kv_mb)
    axes[0].plot(seq_lengths, attn_mems, color=color, linestyle=ls, marker='o',
                 markersize=5, label=variant)
    axes[1].plot(seq_lengths, kv_mems, color=color, linestyle=ls, marker='o',
                 markersize=5, label=variant)

for ax, title, ylabel in zip(axes,
    ["Attention Score Matrix Memory (per layer, FP16)", "KV Cache Memory (per layer, FP16)"],
    ["Memory (MB)", "Memory (MB)"]):
    ax.set_xlabel("Sequence length")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_yscale('log')
    ax.set_xscale('log')

plt.suptitle("Memory Scaling: MHA vs GQA vs MQA (d_k=128, 32 Q heads)", fontsize=12)
plt.tight_layout()
plt.savefig("attention_memory_scaling.png", dpi=120, bbox_inches='tight')
plt.show()

print("Key numbers (T=32K, 1 layer, FP16):")
for name, n_kv in [("MHA", 32), ("GQA-8", 4), ("MQA", 1)]:
    attn, kv = estimate_attention_memory(32768, n_heads, d_k, n_kv)
    print(f"  {name}: attn={attn:.0f}MB, kv={kv:.0f}MB")

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| $O(N^2)$ attention | Standard attention materializes $N \times N$ score matrix; bottleneck for long context |
| FlashAttention | Tile attention across SRAM; never write full $N \times N$ to HBM; same output, 2-4x faster |
| FlashAttention key insight | IO-awareness: HBM bandwidth, not FLOPs, is the bottleneck for attention |
| MQA | 1 K,V head for all Q heads; $H$x KV cache reduction; slight quality loss |
| GQA | $G$ K,V heads, $H/G$ Q heads per KV head; tunable tradeoff; current standard |
| LLaMA-2/3 GQA | 8 KV heads for 70B model; 8x KV cache vs MHA with minor quality impact |
| KV cache formula | $2 \times L \times H_{\text{kv}} \times T \times d_k \times \text{bytes}$ |
| Sliding window attn | Each token attends to window of size $W$; $O(NW)$ instead of $O(N^2)$; Mistral |
| Linear attention | Kernel approximation to softmax attention; $O(N)$ but lower quality in practice |